In [1]:
#collect_sysinfo.py
import json
import psutil
import platform
import socket


def collect_system_info():
    data = {}

    # CPU info
    data["cpu"] = {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_processors": psutil.cpu_count(logical=True),
        "cpu_freq": psutil.cpu_freq()._asdict() if psutil.cpu_freq() else None,
        "cpu_percent_total": psutil.cpu_percent(interval=0.1),
        "cpu_percent_per_cpu": psutil.cpu_percent(interval=0.1, percpu=True),
    }

    # Memory info
    data["memory"] = psutil.virtual_memory()._asdict()
    data["swap"] = psutil.swap_memory()._asdict()

    # Storage info (alle partities)
    storage = {}
    for part in psutil.disk_partitions(all=False):
        try:
            usage = psutil.disk_usage(part.mountpoint)
            storage[part.mountpoint] = {
                "device": part.device,
                "fstype": part.fstype,
                "opts": part.opts,
                "total": usage.total,
                "used": usage.used,
                "free": usage.free,
                "percent": usage.percent,
            }
        except PermissionError:
            continue
    data["storage"] = storage

    # Network info
    net = {}
    addrs = psutil.net_if_addrs()
    stats = psutil.net_if_stats()
    io = psutil.net_io_counters(pernic=True)
    for iface, addr_list in addrs.items():
        net[iface] = {
            "addrs": [a._asdict() for a in addr_list],
            "stats": stats[iface]._asdict() if iface in stats else None,
            "io": io[iface]._asdict() if iface in io else None,
        }

    data["network"] = {
        "hostname": socket.gethostname(),
        "fqdn": socket.getfqdn(),
        "interfaces": net,
    }

    return data


if __name__ == "__main__":
    info = collect_system_info()
    # JSON output
    print(json.dumps(info, indent=2))

{
  "cpu": {
    "platform": "Windows-11-10.0.26200-SP0",
    "processor": "Intel64 Family 6 Model 154 Stepping 3, GenuineIntel",
    "physical_cores": 8,
    "logical_processors": 12,
    "cpu_freq": {
      "current": 2000.0,
      "min": 0.0,
      "max": 2000.0
    },
    "cpu_percent_total": 24.6,
    "cpu_percent_per_cpu": [
      0.0,
      0.0,
      14.3,
      0.0,
      0.0,
      0.0,
      33.3,
      0.0,
      16.7,
      100.0,
      16.7,
      28.6
    ]
  },
  "memory": {
    "total": 16866672640,
    "available": 2322583552,
    "percent": 86.2,
    "used": 14544089088,
    "free": 2322583552
  },
  "swap": {
    "total": 16866672640,
    "used": 742195200,
    "free": 16124477440,
    "percent": 4.4,
    "sin": 0,
    "sout": 0
  },
  "storage": {
    "C:\\": {
      "device": "C:\\",
      "fstype": "NTFS",
      "opts": "rw,fixed",
      "total": 485510606848,
      "used": 443832324096,
      "free": 41678282752,
      "percent": 91.4
    }
  },
  "network": {
 